In [1]:
import pandas as pd

# The 6 indicator codes we're focusing on
codes = ['IT.NET.USER.ZS', 'IT.NET.USER.FE.ZS', 'IT.NET.USER.MA.ZS', 
         'IT.CEL.SETS.P2', 'IT.NET.BBND.P2', 'IT.MLT.MAIN.P2']

# Read the file in chunks (since it's 189MB, this avoids memory issues)
chunks = []
for chunk in pd.read_csv('WDICSV.csv', chunksize=50000, dtype=str):
    filtered = chunk[chunk['Indicator Code'].isin(codes)]
    if len(filtered) > 0:
        chunks.append(filtered)

df = pd.concat(chunks, ignore_index=True)
print("Shape:", df.shape)
df.head()

Shape: (1590, 70)


,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,...,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,Africa Eastern and Southern,AFE,Fixed broadband subscriptions (per 100 people),IT.NET.BBND.P2,NaN,NaN,NaN,NaN,NaN,NaN,...,0.5,0.5,0.5,0.6,0.51,0.62,0.67,0.78,1.03,1.41
1,Africa Eastern and Southern,AFE,Fixed telephone subscriptions (per 100 people),IT.MLT.MAIN.P2,0.8370045954602099,0.8423679892105397,0.843302196219782,0.844147336182397,0.8448892278053246,0.8024198547330943,...,1.3,1.3,1,0.8,0.7,0.6,0.6,0.5,0.5,0.4
2,Africa Eastern and Southern,AFE,Individuals using the Internet (% of population),IT.NET.USER.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,16.3,17.3,19.6,21.6,23.5,25,26.8,27.8,28.8,30.4
3,Africa Eastern and Southern,AFE,"Individuals using the Internet, female (% of f...",IT.NET.USER.FE.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,18.4,20,21.9,23.8,24.8,25.6,NaN
4,Africa Eastern and Southern,AFE,"Individuals using the Internet, male (% of mal...",IT.NET.USER.MA.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,24.9,26.2,28.2,29.8,30.9,32,NaN


In [2]:
# Identify which columns are "identifier" columns vs "year" columns
id_cols = ['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code']
year_cols = [c for c in df.columns if c not in id_cols]

# Melt from wide to long format
df_long = df.melt(id_vars=id_cols, value_vars=year_cols, var_name='Year', value_name='Value')

# Drop rows where there's no data at all
df_long = df_long.dropna(subset=['Value'])

# Convert Year and Value to proper number types
df_long['Year'] = df_long['Year'].astype(int)
df_long['Value'] = df_long['Value'].astype(float)

print("Long format shape:", df_long.shape)
df_long.head(10)

Long format shape: (42880, 6)


,Country Name,Country Code,Indicator Name,Indicator Code,Year,Value
1,Africa Eastern and Southern,AFE,Fixed telephone subscriptions (per 100 people),IT.MLT.MAIN.P2,1960,0.837005
5,Africa Eastern and Southern,AFE,Mobile cellular subscriptions (per 100 people),IT.CEL.SETS.P2,1960,0.000000
7,Africa Western and Central,AFW,Fixed telephone subscriptions (per 100 people),IT.MLT.MAIN.P2,1960,0.060733
11,Africa Western and Central,AFW,Mobile cellular subscriptions (per 100 people),IT.CEL.SETS.P2,1960,0.000000
17,Arab World,ARB,Mobile cellular subscriptions (per 100 people),IT.CEL.SETS.P2,1960,0.000000
23,Caribbean small states,CSS,Mobile cellular subscriptions (per 100 people),IT.CEL.SETS.P2,1960,0.000000
29,Central Europe and the Baltics,CEB,Mobile cellular subscriptions (per 100 people),IT.CEL.SETS.P2,1960,0.000000
31,Early-demographic dividend,EAR,Fixed telephone subscriptions (per 100 people),IT.MLT.MAIN.P2,1960,0.402379
35,Early-demographic dividend,EAR,Mobile cellular subscriptions (per 100 people),IT.CEL.SETS.P2,1960,0.000000
41,East Asia & Pacific,EAS,Mobile cellular subscriptions (per 100 people),IT.CEL.SETS.P2,1960,0.000000


In [3]:
# Load the country metadata file
df_country = pd.read_csv('WDICountry.csv', dtype=str)

# Keep only rows that have a Region assigned - these are real countries
real_countries = df_country[df_country['Region'].notna()][['Country Code', 'Region', 'Income Group']]

print("Real countries found:", len(real_countries))
real_countries.head()

Real countries found: 217


,Country Code,Region,Income Group
0,ABW,Latin America & Caribbean,High income
2,AFG,Middle East & North Africa,Low income
4,AGO,Sub-Saharan Africa,Lower middle income
5,ALB,Europe & Central Asia,Upper middle income
6,AND,Europe & Central Asia,High income


In [4]:
# Merge - this keeps only rows where the country code matches a REAL country
df_clean = df_long.merge(real_countries, on='Country Code', how='inner')

print("Before cleaning:", df_long.shape)
print("After keeping only real countries:", df_clean.shape)
print("Unique countries remaining:", df_clean['Country Name'].nunique())
df_clean.head(10)

Before cleaning: (42880, 6)
After keeping only real countries: (35667, 8)
Unique countries remaining: 214


,Country Name,Country Code,Indicator Name,Indicator Code,Year,Value,Region,Income Group
0,Afghanistan,AFG,Fixed telephone subscriptions (per 100 people),IT.MLT.MAIN.P2,1960,0.089302,Middle East & North Africa,Low income
1,Afghanistan,AFG,Mobile cellular subscriptions (per 100 people),IT.CEL.SETS.P2,1960,0.000000,Middle East & North Africa,Low income
2,Albania,ALB,Fixed telephone subscriptions (per 100 people),IT.MLT.MAIN.P2,1960,0.400014,Europe & Central Asia,Upper middle income
3,Albania,ALB,Mobile cellular subscriptions (per 100 people),IT.CEL.SETS.P2,1960,0.000000,Europe & Central Asia,Upper middle income
4,Algeria,DZA,Mobile cellular subscriptions (per 100 people),IT.CEL.SETS.P2,1960,0.000000,Middle East & North Africa,Upper middle income
5,American Samoa,ASM,Mobile cellular subscriptions (per 100 people),IT.CEL.SETS.P2,1960,0.000000,East Asia & Pacific,High income
6,Andorra,AND,Mobile cellular subscriptions (per 100 people),IT.CEL.SETS.P2,1960,0.000000,Europe & Central Asia,High income
7,Angola,AGO,Fixed telephone subscriptions (per 100 people),IT.MLT.MAIN.P2,1960,0.124431,Sub-Saharan Africa,Lower middle income
8,Angola,AGO,Mobile cellular subscriptions (per 100 people),IT.CEL.SETS.P2,1960,0.000000,Sub-Saharan Africa,Lower middle income
9,Antigua and Barbuda,ATG,Mobile cellular subscriptions (per 100 people),IT.CEL.SETS.P2,1960,0.000000,Latin America & Caribbean,High income


In [5]:
print("Year range:", df_clean['Year'].min(), "-", df_clean['Year'].max())
print()
print("Number of data points per year (last 15 years):")
print(df_clean.groupby('Year').size().tail(15))

Year range: 1960 - 2025

Number of data points per year (last 15 years):
Year
2011    903
2012    935
2013    950
2014    964
2015    975
2016    954
2017    984
2018    905
2019    991
2020    947
2021    951
2022    949
2023    821
2024    760
2025     24
dtype: int64


In [6]:
# Filter to 2000-2022
df_final = df_clean[(df_clean['Year'] >= 2000) & (df_clean['Year'] <= 2022)]

print("Final shape:", df_final.shape)
print("Year range:", df_final['Year'].min(), "-", df_final['Year'].max())
print("Countries:", df_final['Country Name'].nunique())

# Save the final cleaned dataset
df_final.to_csv('WDI_Tech_Clean_Final.csv', index=False)
print("\nSaved as WDI_Tech_Clean_Final.csv")

Final shape: (20404, 8)
Year range: 2000 - 2022
Countries: 214

Saved as WDI_Tech_Clean_Final.csv


In [7]:
# Filter to just the "Internet users (% of population)" indicator
internet_df = df_final[df_final['Indicator Code'] == 'IT.NET.USER.ZS']

# Average internet usage % across all countries, per year
yearly_avg = internet_df.groupby('Year')['Value'].mean()

print("Average global internet usage % by year:")
print(yearly_avg)

Average global internet usage % by year:
Year
2000     8.732304
2001    10.653857
2002    13.617631
2003    16.190692
2004    18.613688
2005    20.692561
2006    23.310742
2007    25.723131
2008    28.537209
2009    31.267055
2010    34.354965
2011    37.093038
2012    39.634400
2013    42.188204
2014    45.196785
2015    47.857891
2016    51.483996
2017    55.442172
2018    56.998479
2019    59.745547
2020    64.107147
2021    68.001652
2022    70.243193
Name: Value, dtype: float64


In [9]:
# Focus on the most recent year (2022) and compare by Region
internet_2022 = df_final[(df_final['Indicator Code'] == 'IT.NET.USER.ZS') & (df_final['Year'] == 2022)]

region_avg = internet_2022.groupby('Region')['Value'].mean().sort_values(ascending=False)

print("Average internet usage % by Region (2022):")
print(region_avg)

Average internet usage % by Region (2022):
Region
North America                 93.364250
Europe & Central Asia         88.118045
Middle East & North Africa    83.928153
East Asia & Pacific           76.085503
Latin America & Caribbean     74.257098
South Asia                    60.589232
Sub-Saharan Africa            38.147439
Name: Value, dtype: float64


In [10]:
# Compare female vs male internet usage globally, most recent year
female_2022 = df_final[(df_final['Indicator Code'] == 'IT.NET.USER.FE.ZS') & (df_final['Year'] == 2022)]['Value'].mean()
male_2022 = df_final[(df_final['Indicator Code'] == 'IT.NET.USER.MA.ZS') & (df_final['Year'] == 2022)]['Value'].mean()

print(f"Average female internet usage (2022): {female_2022:.2f}%")
print(f"Average male internet usage (2022): {male_2022:.2f}%")
print(f"Gap: {male_2022 - female_2022:.2f} percentage points")

Average female internet usage (2022): 85.45%
Average male internet usage (2022): 86.77%
Gap: 1.32 percentage points


In [11]:
# Female vs Male internet usage by Income Group, 2022
female_by_income = df_final[(df_final['Indicator Code'] == 'IT.NET.USER.FE.ZS') & (df_final['Year'] == 2022)].groupby('Income Group')['Value'].mean()
male_by_income = df_final[(df_final['Indicator Code'] == 'IT.NET.USER.MA.ZS') & (df_final['Year'] == 2022)].groupby('Income Group')['Value'].mean()

gap_by_income = (male_by_income - female_by_income).sort_values(ascending=False)

print("Female internet usage % by Income Group (2022):")
print(female_by_income)
print("\nMale internet usage % by Income Group (2022):")
print(male_by_income)
print("\nGender gap (percentage points) by Income Group:")
print(gap_by_income)

Female internet usage % by Income Group (2022):
Income Group
High income            90.479845
Lower middle income    67.051525
Upper middle income    80.554004
Name: Value, dtype: float64

Male internet usage % by Income Group (2022):
Income Group
High income            91.24276
Lower middle income    74.62640
Upper middle income    81.76275
Name: Value, dtype: float64

Gender gap (percentage points) by Income Group:
Income Group
Lower middle income    7.574875
Upper middle income    1.208746
High income            0.762915
Name: Value, dtype: float64
